# WTI COT MM Nowcasting — Kalman Filter — 04 Model Comparison

Unified **walk-forward benchmarking** of all models on the same date range and split points.

| ID | Model | Features |
|---|---|---|
| A | Local Level (diagonal) | None |
| B | Local Level (full cov) | None |
| C | Local Linear Trend | None |
| D | Local Level + Fixed Regression | 5 selected |
| E | Dynamic Regression | 5 selected |
| OLS | Ordinary Least Squares | 5 selected |
| Ridge | Ridge Regression (α via CV) | 5 selected |

**Primary metric:** Spearman ρ between predicted and actual position changes (consistent with the existing ML pipeline).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../../../')

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.linear_model import Ridge, LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from statsmodels.tsa.statespace.mlemodel import MLEModel

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

OUT_DIR = pathlib.Path('../../../cache/output/wti/mm')

In [ ]:
from src.utils.io.read import PreprocessedDataReader
from src.preprocessing.base import FutureTicker
from src.settings import Settings

pdr = PreprocessedDataReader(Settings.historical.paths.PREPROCESSED_DATA_PATH)
dataset = pdr.read_dataset(ticker=FutureTicker.WTI)
dataset['tradeDate'] = pd.to_datetime(dataset['tradeDate'])
dataset.sort_values('tradeDate', inplace=True)
dataset.reset_index(drop=True, inplace=True)

CONFIG_PATH = OUT_DIR / 'kf_config.json'
with open(CONFIG_PATH) as f:
    kf_config = json.load(f)

RESPONSES = kf_config['responses_raw']
FEATURES  = kf_config['selected_features']
RESP_LABELS = ['Net', 'Long', 'Short']

---
## 1. Shared Walk-Forward Setup

All models are evaluated on the **same joint-clean dataset** (rows where both responses and features are non-NaN) with **identical split points**.

In [ ]:
cols_needed = ['tradeDate'] + RESPONSES + FEATURES
df = dataset[cols_needed].dropna().reset_index(drop=True)

dates   = df['tradeDate'].values
Y_raw   = df[RESPONSES].values.astype(float)   # (T, 3) original scale
X_raw   = df[FEATURES].values.astype(float)    # (T, 5) original scale

Y_mean, Y_std = Y_raw.mean(axis=0), Y_raw.std(axis=0)
X_mean, X_std = X_raw.mean(axis=0), X_raw.std(axis=0)
Y = (Y_raw - Y_mean) / Y_std
X = (X_raw - X_mean) / X_std

MIN_TRAIN   = 200
REFIT_EVERY = 26

OOS_START = MIN_TRAIN + 1
OOS_DATES = dates[OOS_START:]

print(f'Total obs      : {len(df)}')
print(f'Min train      : {MIN_TRAIN}')
print(f'OOS window     : {len(df) - OOS_START} observations')
print(f'OOS period     : {pd.Timestamp(OOS_DATES[0]).date()} → {pd.Timestamp(OOS_DATES[-1]).date()}')

---
## 2. Linear Model Walk-Forward (OLS & Ridge)

For each step `t`, we fit **one model per response** on `(X[:t], y_i[:t])` and predict `X[t+1]`.  
Scaling is done inside each training window (no data leakage).

In [ ]:
def walk_forward_linear(model_fn, Y_raw, X_raw, min_train, refit_every):
    """
    Walk-forward for univariate-per-response linear models.
    model_fn: callable that returns a sklearn-compatible estimator.
    Returns a DataFrame of predictions vs actuals in original scale.
    """
    T = len(Y_raw)
    k = Y_raw.shape[1]
    preds = np.full((T - min_train - 1, k), np.nan)
    estimators = [None] * k

    for t in range(min_train, T - 1):
        X_train, Y_train = X_raw[:t], Y_raw[:t]
        refit = (estimators[0] is None) or ((t - min_train) % refit_every == 0)

        for i in range(k):
            if refit:
                pipe = Pipeline([
                    ('scaler', StandardScaler()),
                    ('model',  model_fn()),
                ])
                pipe.fit(X_train, Y_train[:, i])
                estimators[i] = pipe

            preds[t - min_train, i] = estimators[i].predict(X_raw[t + 1:t + 2])[0]

    actuals = Y_raw[min_train + 1:]
    return pd.DataFrame({
        'date':         dates[min_train + 1:],
        'pred_net':     preds[:, 0],
        'pred_long':    preds[:, 1],
        'pred_short':   preds[:, 2],
        'actual_net':   actuals[:, 0],
        'actual_long':  actuals[:, 1],
        'actual_short': actuals[:, 2],
    })

print('Running walk-forward for OLS...')
wf_ols = walk_forward_linear(LinearRegression, Y_raw, X_raw, MIN_TRAIN, REFIT_EVERY)

print('Running walk-forward for Ridge (RidgeCV α)...')
wf_ridge = walk_forward_linear(
    lambda: RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5),
    Y_raw, X_raw, MIN_TRAIN, REFIT_EVERY
)

print('Done.')

---
## 3. Load KF Walk-Forward Results

Loaded from CSVs saved in notebooks 02 and 03.

In [ ]:
wf_A = pd.read_csv(OUT_DIR / 'kf_wf_baseline_A.csv', parse_dates=['date'])
wf_B = pd.read_csv(OUT_DIR / 'kf_wf_baseline_B.csv', parse_dates=['date'])
wf_C = pd.read_csv(OUT_DIR / 'kf_wf_baseline_C.csv', parse_dates=['date'])
wf_D = pd.read_csv(OUT_DIR / 'kf_wf_model_D.csv',    parse_dates=['date'])
wf_E = pd.read_csv(OUT_DIR / 'kf_wf_model_E.csv',    parse_dates=['date'])

# Align all on the common OOS date index (inner join)
all_wf = {
    'A: LL-diag':      wf_A,
    'B: LL-full':      wf_B,
    'C: LLT':          wf_C,
    'D: LL+Feat':      wf_D,
    'E: Dyn-Reg':      wf_E,
    'OLS':             wf_ols,
    'Ridge':           wf_ridge,
}

for name, wf in all_wf.items():
    print(f'{name:18s}  {len(wf):4d} OOS obs  '
          f'{pd.Timestamp(wf["date"].min()).date()} → {pd.Timestamp(wf["date"].max()).date()}')

---
## 4. Metrics Computation

In [ ]:
def compute_metrics(wf_df):
    rows = []
    for r, label in zip(['net', 'long', 'short'], RESP_LABELS):
        pred   = wf_df[f'pred_{r}'].values
        actual = wf_df[f'actual_{r}'].values
        mask   = ~(np.isnan(pred) | np.isnan(actual))
        if mask.sum() < 10:
            rows.append({'Response': label, 'Spearman ρ': np.nan,
                         'RMSE': np.nan, 'Dir. Acc': np.nan, 'n': mask.sum()})
            continue
        rho, _   = stats.spearmanr(pred[mask], actual[mask])
        rmse     = np.sqrt(np.mean((pred[mask] - actual[mask]) ** 2))
        dir_acc  = np.mean(np.sign(pred[mask]) == np.sign(actual[mask]))
        rows.append({'Response': label, 'Spearman ρ': rho,
                     'RMSE': rmse, 'Dir. Acc': dir_acc, 'n': int(mask.sum())})
    return pd.DataFrame(rows).set_index('Response')

metrics_all = {name: compute_metrics(wf) for name, wf in all_wf.items()}

In [ ]:
# Master metrics table — Spearman ρ
spearman_table = pd.concat(
    [m[['Spearman ρ']].rename(columns={'Spearman ρ': name})
     for name, m in metrics_all.items()],
    axis=1,
).round(4)

print('=== Spearman ρ (OOS) ===')
spearman_table

In [ ]:
rmse_table = pd.concat(
    [m[['RMSE']].rename(columns={'RMSE': name})
     for name, m in metrics_all.items()],
    axis=1,
).round(1)

print('=== RMSE (original scale, contracts) ===')
rmse_table

In [ ]:
dir_table = pd.concat(
    [m[['Dir. Acc']].rename(columns={'Dir. Acc': name})
     for name, m in metrics_all.items()],
    axis=1,
).round(4)

print('=== Directional Accuracy ===')
dir_table

---
## 5. Visualisations

### 5.1 Spearman ρ Bar Chart

In [ ]:
model_names = list(all_wf.keys())
colors = [
    '#aec6e8', '#aec6e8', '#aec6e8',   # baselines (A, B, C) — light blue
    '#1f77b4', '#ff7f0e',               # KF+feat (D, E) — blue, orange
    '#2ca02c', '#9467bd',               # linear (OLS, Ridge) — green, purple
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, resp in zip(axes, RESP_LABELS):
    rhos = [metrics_all[m].loc[resp, 'Spearman ρ'] for m in model_names]
    bars = ax.bar(model_names, rhos, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{resp} Position Change', fontweight='bold')
    ax.set_ylabel('OOS Spearman ρ')
    ax.set_ylim(min(-0.05, min(rhos) - 0.05), max(rhos) + 0.08)
    ax.set_xticklabels(model_names, rotation=40, ha='right', fontsize=8)
    for bar, rho in zip(bars, rhos):
        if not np.isnan(rho):
            ax.text(bar.get_x() + bar.get_width() / 2, rho + 0.005,
                    f'{rho:.3f}', ha='center', va='bottom', fontsize=7.5)

fig.suptitle('OOS Walk-Forward Spearman ρ — All Models', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Rolling 52-Week Spearman ρ — Net Position

In [ ]:
WINDOW = 52

line_styles = {
    'A: LL-diag':  ('grey',      '--', 1.0),
    'B: LL-full':  ('silver',    '--', 1.0),
    'C: LLT':      ('darkgrey',  '--', 1.0),
    'D: LL+Feat':  ('#1f77b4',   '-',  1.8),
    'E: Dyn-Reg':  ('#ff7f0e',   '-',  1.8),
    'OLS':         ('#2ca02c',   '-',  1.8),
    'Ridge':       ('#9467bd',   '-',  1.8),
}

fig, ax = plt.subplots(figsize=(14, 5))

for name, wf in all_wf.items():
    pred   = wf['pred_net'].values
    actual = wf['actual_net'].values
    mask   = ~(np.isnan(pred) | np.isnan(actual))
    pred, actual = pred[mask], actual[mask]
    wf_dates = wf['date'].values[mask]

    rolling = [
        stats.spearmanr(pred[max(0, i-WINDOW):i],
                        actual[max(0, i-WINDOW):i]).statistic
        if i >= WINDOW else np.nan
        for i in range(len(pred))
    ]
    color, ls, lw = line_styles[name]
    ax.plot(wf_dates, rolling, label=name, color=color, linestyle=ls, linewidth=lw)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'Rolling {WINDOW}-Week OOS Spearman ρ — Net Position Change')
ax.set_ylabel('Spearman ρ')
ax.set_xlabel('Date')
ax.legend(ncol=4, fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

### 5.3 Error Distribution Comparison

In [ ]:
# Show error distributions for models with features only (D, E, OLS, Ridge)
feat_models = {'D: LL+Feat': wf_D, 'E: Dyn-Reg': wf_E, 'OLS': wf_ols, 'Ridge': wf_ridge}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, r, label in zip(axes, ['net', 'long', 'short'], RESP_LABELS):
    for name, wf in feat_models.items():
        resid = (wf[f'actual_{r}'] - wf[f'pred_{r}']).dropna()
        resid.plot.kde(ax=ax, label=name)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{label} — OOS Residual Distribution')
    ax.set_xlabel('Actual − Predicted (contracts)')
    ax.legend(fontsize=8)
plt.suptitle('OOS Residual Distributions — Feature Models', y=1.02)
plt.tight_layout()
plt.show()

### 5.4 Predicted vs Actual Scatter — Net (Best Models)

In [ ]:
best_models = {
    'D: LL+Feat':  (wf_D,    '#1f77b4'),
    'E: Dyn-Reg':  (wf_E,    '#ff7f0e'),
    'OLS':         (wf_ols,  '#2ca02c'),
    'Ridge':       (wf_ridge,'#9467bd'),
}

fig, axes = plt.subplots(1, len(best_models), figsize=(16, 4))
for ax, (name, (wf, color)) in zip(axes, best_models.items()):
    pred   = wf['pred_net'].values
    actual = wf['actual_net'].values
    mask   = ~(np.isnan(pred) | np.isnan(actual))
    rho, _ = stats.spearmanr(pred[mask], actual[mask])
    ax.scatter(actual[mask], pred[mask], s=6, alpha=0.3, color=color)
    lim = np.percentile(np.abs(np.concatenate([actual[mask], pred[mask]])), 98)
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_title(f'{name}\nρ = {rho:.3f}', fontsize=10)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
plt.suptitle('Predicted vs Actual — Net Position Change (OOS)', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Statistical Significance — Diebold-Mariano Test

We test whether prediction errors of a candidate model are **statistically different** from the baseline (Model A) using a Diebold-Mariano test (Harvey, Leybourne, Newbold 1997 correction).

In [ ]:
def diebold_mariano(e1, e2, h=1):
    """
    DM test: H0 = equal predictive accuracy.
    e1, e2 : forecast error arrays (actual - predicted)
    h      : forecast horizon (1 for 1-step ahead)
    Returns: DM statistic, p-value (two-sided)
    """
    d = e1 ** 2 - e2 ** 2   # loss differential (MSE-based)
    T = len(d)
    d_bar = np.mean(d)
    # HAC variance with h-1 autocovariances
    gamma = [np.cov(d[lag:], d[:T-lag])[0, 1] if lag > 0 else np.var(d, ddof=1)
             for lag in range(h)]
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / T
    # Harvey et al. correction
    corr = np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    dm_stat = (d_bar / np.sqrt(max(var_d, 1e-12))) * corr
    p_val   = 2 * stats.t.sf(np.abs(dm_stat), df=T - 1)
    return dm_stat, p_val


# Baseline is Model A
baseline_name = 'A: LL-diag'
wf_base = all_wf[baseline_name]

dm_rows = []
for resp, r in zip(RESP_LABELS, ['net', 'long', 'short']):
    e_base = (wf_base[f'actual_{r}'] - wf_base[f'pred_{r}']).dropna().values

    for name, wf in all_wf.items():
        if name == baseline_name:
            continue
        e_cand = (wf[f'actual_{r}'] - wf[f'pred_{r}']).dropna().values

        # Align lengths
        n = min(len(e_base), len(e_cand))
        dm_stat, p_val = diebold_mariano(e_base[-n:], e_cand[-n:])

        dm_rows.append({
            'Response':  resp,
            'Model':     name,
            'DM stat':   round(dm_stat, 3),
            'p-value':   round(p_val, 4),
            'Better than baseline?': 'Yes *' if (p_val < 0.05 and dm_stat > 0) else
                                     ('Yes' if dm_stat > 0 else 'No'),
        })

dm_df = pd.DataFrame(dm_rows)
dm_df.pivot(index='Model', columns='Response', values='p-value').round(4)

In [ ]:
print('Diebold-Mariano Results (DM stat > 0 → candidate beats baseline):')
print(dm_df.to_string(index=False))

---
## 7. Regime Analysis

Do models differ more in trending vs mean-reverting regimes?  
We proxy regimes using the **rolling 26-week z-score** of net MM position.

In [ ]:
# Compute regime indicator from net position z-score
net_series = dataset.set_index('tradeDate')['ManagedMoney_NetPosition'].dropna()
rolling_z = (net_series - net_series.rolling(52).mean()) / net_series.rolling(52).std()

# Map to OOS dates
rolling_z_oos = rolling_z.reindex(pd.to_datetime(wf_D['date'])).values

extreme_long  = rolling_z_oos >  1.5   # crowded long
extreme_short = rolling_z_oos < -1.5   # crowded short
neutral       = np.abs(rolling_z_oos) <= 1.5

print(f'Extreme long  : {extreme_long.sum()} weeks ({100*extreme_long.mean():.1f}%)')
print(f'Extreme short : {extreme_short.sum()} weeks ({100*extreme_short.mean():.1f}%)')
print(f'Neutral       : {neutral.sum()} weeks ({100*neutral.mean():.1f}%)')

In [ ]:
# Spearman ρ per regime for each model (Net response)
regime_labels = {'Extreme long': extreme_long, 'Neutral': neutral, 'Extreme short': extreme_short}

regime_rows = []
for model_name, wf in all_wf.items():
    pred   = wf['pred_net'].values
    actual = wf['actual_net'].values

    for regime_label, mask_base in regime_labels.items():
        n = min(len(pred), len(mask_base))
        mask = mask_base[:n] & ~(np.isnan(pred[:n]) | np.isnan(actual[:n]))
        if mask.sum() < 10:
            rho = np.nan
        else:
            rho, _ = stats.spearmanr(pred[:n][mask], actual[:n][mask])
        regime_rows.append({'Model': model_name, 'Regime': regime_label,
                            'Spearman ρ': round(rho, 4), 'n': mask.sum()})

regime_df = pd.DataFrame(regime_rows)
regime_df.pivot(index='Model', columns='Regime', values='Spearman ρ').round(4)

In [ ]:
# Bar chart: regime-conditional Spearman ρ
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, regime_label in regime_labels.keys() | regime_labels.keys():
    pass

for ax, regime_label in zip(axes, regime_labels.keys()):
    sub = regime_df[regime_df['Regime'] == regime_label]
    bar_colors = [
        '#aec6e8' if m in ('A: LL-diag', 'B: LL-full', 'C: LLT') else
        '#1f77b4' if m == 'D: LL+Feat' else
        '#ff7f0e' if m == 'E: Dyn-Reg' else
        '#2ca02c' if m == 'OLS' else '#9467bd'
        for m in sub['Model']
    ]
    ax.bar(sub['Model'], sub['Spearman ρ'], color=bar_colors, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{regime_label}\n(n={int(sub["n"].iloc[0])})')
    ax.set_xticklabels(sub['Model'], rotation=40, ha='right', fontsize=8)
    ax.set_ylabel('Spearman ρ')

plt.suptitle('Regime-Conditional OOS Spearman ρ — Net Position Change', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Summary Scorecard

In [ ]:
# Rank models by mean Spearman ρ across the 3 responses
mean_rho = spearman_table.mean(axis=0).rename('Mean Spearman ρ')
mean_rmse = rmse_table.mean(axis=0).rename('Mean RMSE')
mean_dir  = dir_table.mean(axis=0).rename('Mean Dir. Acc')

scorecard = pd.concat([mean_rho, mean_rmse, mean_dir], axis=1)
scorecard['Rank (ρ)'] = scorecard['Mean Spearman ρ'].rank(ascending=False).astype(int)
scorecard = scorecard.sort_values('Rank (ρ)')
scorecard.round(4)

In [ ]:
# Highlight the winner per response
best_per_response = spearman_table.idxmax(axis=1)
print('Best model per response (Spearman ρ):')
print(best_per_response.to_string())
print()
print('Overall recommendation:', scorecard.index[0])

In [ ]:
# Save scorecard and best model info
scorecard.to_csv(OUT_DIR / 'kf_model_comparison_scorecard.csv')

best_model_name = scorecard.index[0]
result = {
    'best_model': best_model_name,
    'scorecard': scorecard.round(4).to_dict(),
    'best_per_response': best_per_response.to_dict(),
}
with open(OUT_DIR / 'kf_comparison_results.json', 'w') as f:
    json.dump(result, f, indent=2)

print(f'Results saved. Best model: {best_model_name}')

---
## 9. Key Takeaways

*(To be filled after running the notebook)*

**Signal from features:**
- Does adding features (D, E) materially improve over the baseline (A)?
- Is the improvement statistically significant (DM test)?

**KF vs linear models:**
- Does the latent state provide information beyond what OLS/Ridge capture directly?
- KF advantage is most likely in **regime transitions** where the latent state adapts faster

**Dynamic vs fixed coefficients:**
- Does Model E (time-varying β) outperform Model D (fixed β)?
- If yes: feature sensitivities are non-stationary → worth modelling explicitly
- If no: fixed β is sufficient, prefer Model D for interpretability

**Carry forward to Notebook 05:** The winning model by mean Spearman ρ, fitted on the full sample.